### Variant calling module

**CMM262, Winter 2026**

Kyle Gaulton, kgaulton@health.ucsd.edu
<br>
<br>

<b>In this walkthrough we will be functionally annotating variant calls from 23andme</b>

<br>
<b><u>Download and format 23andme file</u></b>
<br><br>
For the purposes of this walkthrough, the Harvard Personal Genome Project has publicly available genetic data from many individuals:   
https://my.pgp-hms.org/public_genetic_data  
<br><br>
We will use 23andme genetic data from one of the individuals in this database
<br><br>
If you have used 23andme you could download your genotype data file directly and annotate your own variants instead using the same approach
<br><br>

In [1]:
cp ~/public/variantcalling/resources/23andme_indiv1.txt .

<br>
If we look at the file we can see that it isn't in a standard (e.g. VCF) format, but just lists the variants and the genotype

In [2]:
head -n 50 23andme_indiv1.txt

# This data file generated by 23andMe at: Mon Mar 27 07:52:35 2017
#
# This file contains raw genotype data, including data that is not used in 23andMe reports.
# This data has undergone a general quality review however only a subset of markers have been 
# individually validated for accuracy. As such, this data is suitable only for research, 
# educational, and informational use and not for medical or other use.
# 
# Below is a text version of your data.  Fields are TAB-separated
# Each line corresponds to a single SNP.  For each SNP, we provide its identifier 
# (an rsid or an internal id), its location on the reference human genome, and the 
# genotype call oriented with respect to the plus strand on the human reference sequence.
# We are using reference human assembly build 37 (also known as Annotation Release 104).
# Note that it is possible that data downloaded at different times may be different due to ongoing 
# improvements in our ability to call genotypes. More information ab

<br>
Therefore, before annotating the variant calls we need to first convert the 23andme output to a VCF
<br><br>
We will use a Perl script '23andme2vcf.pl' to convert the file to VCF

In [5]:
cp ~/public/variantcalling/resources/23andme_v4_hg19_ref.txt.gz .

In [7]:
perl ~/public/variantcalling/resources/23andme2vcf.pl 23andme_indiv1.txt my_vars.vcf 4

8030 sites were not included; these unmatched references can be found in sites_not_in_reference.txt.Try running again, but specify the other reference version:
./23andme2vcf.pl 23andme_indiv1.txt my_vars.vcf 3


In [8]:
head -n 50 my_vars.vcf

##fileformat=VCFv4.2
##fileDate=20260219
##source=23andme2vcf.pl https://github.com/arrogantrobot/23andme2vcf
##reference=file://23andme_v4_hg19_ref.txt.gz
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	GENOTYPE
chr1	734462	rs12564807	G	A	.	.	.	GT	1/1
chr1	752721	rs3131972	A	G	.	.	.	GT	0/1
chr1	760998	rs148828841	C	.	.	.	.	GT	0/0
chr1	776546	rs12124819	A	.	.	.	.	GT	0/0
chr1	787173	rs115093905	G	.	.	.	.	GT	0/0
chr1	798959	rs11240777	g	A	.	.	.	GT	0/1
chr1	824398	rs7538305	a	.	.	.	.	GT	0/0
chr1	838555	rs4970383	c	.	.	.	.	GT	0/0
chr1	846808	rs4475691	C	.	.	.	.	GT	0/0
chr1	854250	rs7537756	A	.	.	.	.	GT	0/0
chr1	861808	rs13302982	A	G	.	.	.	GT	1/1
chr1	864490	rs55678698	C	.	.	.	.	GT	0/0
chr1	871267	i6019299	C	.	.	.	.	GT	0/0
chr1	873558	rs1110052	G	T	.	.	.	GT	1/1
chr1	878697	rs147226614	G	.	.	.	.	GT	0/0
chr1	881843	i6019302	G	.	.	.	.	GT	0/0
chr1	882033	rs2272756	G	.	.	.	.	GT	0/0
chr1	884767	rs67274836	G	.	.	.	.	GT	0/0
chr1	888554	i601

<br>  
An alternate way to convert to VCF is using bcftools:

#/opt/conda/envs/variant_calling/bin/bcftools convert --tsv2vcf 23andme.txt -f Homo_sapiens.GRCh37.dna.primary_assembly.fa.gz -s Sample -o myvcf.gz

<br>
<b><u>Functionally annotate 23andme VCF</u></b>
<br><br>
Next we will functionally annotate variants in the VCF file for effects on protein-coding genes and to identify variants in ClinVar using ANNOVAR 

In [9]:
cp ~/public/variantcalling/resources/annovar.tar.gz .
gunzip annovar.tar.gz
tar -xvf annovar.tar

annovar/
annovar/coding_change.pl
annovar/variants_reduction.pl
annovar/retrieve_seq_from_fasta.pl
annovar/annotate_variation.pl
annovar/example/
annovar/convert2annovar.pl
annovar/table_annovar.pl
annovar/humandb/
annovar/humandb/GRCh37_MT_ensGene.txt
annovar/humandb/hg19_refGeneMrna.fa
annovar/humandb/annovar_downdb.log
annovar/humandb/hg19_example_db_generic.txt
annovar/humandb/hg19_refGeneVersion.txt
annovar/humandb/hg19_gnomad211_exome.txt.gz.gz
annovar/humandb/hg19_refGene.txt
annovar/humandb/hg19_refGeneWithVerMrna.fa
annovar/humandb/hg19_MT_ensGene.txt
annovar/humandb/hg19_refGeneWithVer.txt
annovar/humandb/GRCh37_MT_ensGeneMrna.fa
annovar/humandb/hg19_clinvar_20131105.txt.idx
annovar/humandb/hg19_exac03.txt
annovar/humandb/hg19_clinvar_20131105.txt
annovar/humandb/hg19_gnomad211_exome.txt.idx.gz
annovar/humandb/hg19_example_db_gff3.txt
annovar/humandb/hg19_MT_ensGeneMrna.fa
annovar/humandb/hg19_exac03.txt.idx
annovar/humandb/genometrax-sample-files-gff/
annovar/humandb/hg19_ex

In [10]:
perl annovar/table_annovar.pl my_vars.vcf annovar/humandb/ -buildver hg19 -out annotated -remove -protocol refGene,clinvar_20131105 -operation g,f -nastring . -vcfinput

NOTICE: the --polish argument is set ON automatically (use --nopolish to change this behavior)

NOTICE: Running with system command <convert2annovar.pl  -includeinfo -allsample -withfreq -format vcf4 my_vars.vcf > annotated.avinput>
NOTICE: Finished reading 580557 lines from VCF file
NOTICE: A total of 580551 locus in VCF file passed QC threshold, representing 580551 SNPs (217896 transitions and 53033 transversions) and 0 indels/substitutions
NOTICE: Finished writing allele frequencies based on 580551 SNP genotypes (217896 transitions and 53033 transversions) and 0 indels/substitutions for 1 samples

NOTICE: Running with system command <annovar/table_annovar.pl annotated.avinput annovar/humandb/ -buildver hg19 -outfile annotated -remove -protocol refGene,clinvar_20131105 -operation g,f -nastring . -otherinfo>
NOTICE: the --polish argument is set ON automatically (use --nopolish to change this behavior)
-----------------------------------------------------------------
NOTICE: Processing

<br>
This step should produce both a VCF with the annotations included as well as a text file of variant annotations

In [11]:
ls -la *multianno*

-rw-r--r-- 1 grader-bnfo262-01 root  72887286 Feb 19 06:57 annotated.hg19_multianno.txt
-rw-r--r-- 1 grader-bnfo262-01 root 133435057 Feb 19 06:57 annotated.hg19_multianno.vcf


<br>
If we look at the annotated text file we can see many columns - including some redundant information - so first we want to clean up the file so it is a bit easier to read

In [12]:
head -n 100 annotated.hg19_multianno.vcf

##fileformat=VCFv4.2
##fileDate=20260219
##source=23andme2vcf.pl https://github.com/arrogantrobot/23andme2vcf
##reference=file://23andme_v4_hg19_ref.txt.gz
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##INFO=<ID=ANNOVAR_DATE,Number=1,Type=String,Description="Flag the start of ANNOVAR annotation for one alternative allele">
##INFO=<ID=Func.refGene,Number=.,Type=String,Description="Func.refGene annotation provided by ANNOVAR">
##INFO=<ID=Gene.refGene,Number=.,Type=String,Description="Gene.refGene annotation provided by ANNOVAR">
##INFO=<ID=GeneDetail.refGene,Number=.,Type=String,Description="GeneDetail.refGene annotation provided by ANNOVAR">
##INFO=<ID=ExonicFunc.refGene,Number=.,Type=String,Description="ExonicFunc.refGene annotation provided by ANNOVAR">
##INFO=<ID=AAChange.refGene,Number=.,Type=String,Description="AAChange.refGene annotation provided by ANNOVAR">
##INFO=<ID=clinvar_20131105,Number=.,Type=String,Description="clinvar_20131105 annotation provided by ANNOV

In [13]:
head -n 20 annotated.hg19_multianno.txt

Chr	Start	End	Ref	Alt	Func.refGene	Gene.refGene	GeneDetail.refGene	ExonicFunc.refGene	AAChange.refGene	clinvar_20131105	Otherinfo1	Otherinfo2	Otherinfo3	Otherinfo4	Otherinfo5	Otherinfo6	Otherinfo7	Otherinfo8	Otherinfo9	Otherinfo10	Otherinfo11	Otherinfo12	Otherinfo13
chr1	734462	734462	G	A	intergenic	LOC100288069;FAM87B	dist=20394;dist=18289	.	.	.	1	.	.	chr1	734462	rs12564807	G	A	.	.	.	GT	1/1
chr1	752721	752721	A	G	upstream	FAM87B	dist=30	.	.	.	0.5	.	.	chr1	752721	rs3131972	A	G	.	.	.	GT	0/1
chr1	760998	760998	C	C	downstream	LINC00115	dist=588	.	.	.	0	.	.	chr1	760998	rs148828841	C	.	.	.	.	GT	0/0
chr1	776546	776546	A	A	ncRNA_intronic	LINC01128	.	.	.	.	0	.	.	chr1	776546	rs12124819	A	.	.	.	.	GT	0/0
chr1	787173	787173	G	G	ncRNA_intronic	LINC01128	.	.	.	.	0	.	.	chr1	787173	rs115093905	G	.	.	.	.	GT	0/0
chr1	798959	798959	G	A	intergenic	LINC01128;FAM41C	dist=4133;dist=4492	.	.	.	0.5	.	.	chr1	798959	rs11240777	g	A	.	.	.	GT	0/1
chr1	824398	824398	A	A	intergenic	FAM41C;LINC02593	dist=12216;dist=27

In [14]:
cut -f1,2,4,5,7,9,10,11,17,24 annotated.hg19_multianno.txt > annotated.hg19_multianno.trim.txt

In [16]:
head -n 50 annotated.hg19_multianno.trim.txt

Chr	Start	Ref	Alt	Gene.refGene	ExonicFunc.refGene	AAChange.refGene	clinvar_20131105	Otherinfo6	Otherinfo13
chr1	734462	G	A	LOC100288069;FAM87B	.	.	.	rs12564807	1/1
chr1	752721	A	G	FAM87B	.	.	.	rs3131972	0/1
chr1	760998	C	C	LINC00115	.	.	.	rs148828841	0/0
chr1	776546	A	A	LINC01128	.	.	.	rs12124819	0/0
chr1	787173	G	G	LINC01128	.	.	.	rs115093905	0/0
chr1	798959	G	A	LINC01128;FAM41C	.	.	.	rs11240777	0/1
chr1	824398	A	A	FAM41C;LINC02593	.	.	.	rs7538305	0/0
chr1	838555	C	C	FAM41C;LINC02593	.	.	.	rs4970383	0/0
chr1	846808	C	C	FAM41C;LINC02593	.	.	.	rs4475691	0/0
chr1	854250	A	A	LINC02593	.	.	.	rs7537756	0/0
chr1	861808	A	G	SAMD11	.	.	.	rs13302982	1/1
chr1	864490	C	C	SAMD11	.	.	.	rs55678698	0/0
chr1	871267	C	C	SAMD11	synonymous SNV	SAMD11:NM_152486:exon5:c.C421C:p.R141R	.	i6019299	0/0
chr1	873558	G	T	SAMD11	.	.	.	rs1110052	1/1
chr1	878697	G	G	SAMD11	synonymous SNV	SAMD11:NM_152486:exon12:c.G1629G:p.W543W	.	rs147226614	0/0
chr1	881843	G	G	NOC2L	synonymous SNV	NOC2L:NM_015658:exon15:c.C1742C:p.

<br>
Now looking at the file it is clear that genotypes for all of the variants are provided, including ones which were homozygote for the reference allele.  Therefore we need to filter the file to just those variants which are heterozygote or homozygote non-reference.

In [17]:
grep -v '0/0' annotated.hg19_multianno.trim.txt > annotated.hg19_multianno.trim.nohomref.txt

In [18]:
head -n 20 annotated.hg19_multianno.trim.nohomref.txt

Chr	Start	Ref	Alt	Gene.refGene	ExonicFunc.refGene	AAChange.refGene	clinvar_20131105	Otherinfo6	Otherinfo13
chr1	734462	G	A	LOC100288069;FAM87B	.	.	.	rs12564807	1/1
chr1	752721	A	G	FAM87B	.	.	.	rs3131972	0/1
chr1	798959	G	A	LINC01128;FAM41C	.	.	.	rs11240777	0/1
chr1	861808	A	G	SAMD11	.	.	.	rs13302982	1/1
chr1	873558	G	T	SAMD11	.	.	.	rs1110052	1/1
chr1	889159	A	C	NOC2L	.	.	.	rs13302945	1/1
chr1	891945	A	G	NOC2L	.	.	.	rs13303106	1/1
chr1	894573	G	A	NOC2L	.	.	.	rs13303010	1/1
chr1	909238	G	C	PLEKHN1	nonsynonymous SNV	PLEKHN1:NM_001160184:exon13:c.G1355C:p.R452P,PLEKHN1:NM_001367552:exon14:c.G1496C:p.R499P,PLEKHN1:NM_032129:exon14:c.G1460C:p.R487P	.	i6060381	1/1
chr1	910935	G	A	PERM1;PLEKHN1	.	.	.	rs2340592	0/1
chr1	918384	G	T	PERM1	.	.	.	rs13303118	1/1
chr1	924898	C	A	PERM1;HES4	.	.	.	rs6665000	1/1
chr1	927309	T	C	PERM1;HES4	.	.	.	rs2341362	1/1
chr1	928836	C	T	PERM1;HES4	.	.	.	rs9777703	1/1
chr1	948692	G	A	ISG15	.	.	.	rs2341365	1/1
chr1	948921	T	C	ISG15	.	.	.	rs15842	1/1
chr1	959842	C	T	AG

<br>
Finally, we want to extract variants that may have clinical significance in ClinVar

In [19]:
grep '[=|]pathogenic' annotated.hg19_multianno.trim.nohomref.txt

chr1	12061468	A	G	MFN2	nonsynonymous SNV	MFN2:NM_001127660:exon8:c.A827G:p.Q276R,MFN2:NM_014874:exon9:c.A827G:p.Q276R	CLINSIG=pathogenic;CLNDBN=Hereditary_motor_and_sensory_neuropathy_with_optic_atrophy;CLNACC=RCV000002366.1	i5008699	0/1
chr1	22469381	A	G	WNT4	nonsynonymous SNV	WNT4:NM_030761:exon1:c.T35C:p.L12P	CLINSIG=pathogenic;CLNDBN=Mullerian_aplasia_and_hyperandrogenism;CLNACC=RCV000006691.1	i5001073	0/1
chr1	70904800	G	T	CTH	nonsynonymous SNV	CTH:NM_001190463:exon11:c.G1112T:p.S371I,CTH:NM_153742:exon11:c.G1076T:p.S359I,CTH:NM_001902:exon12:c.G1208T:p.S403I	CLINSIG=pathogenic;CLNDBN=Homocysteine\x2c_total_plasma\x2c_elevated;CLNACC=RCV000003075.1	rs1021737	0/1
chr1	98348885	G	A	DPYD	nonsynonymous SNV	DPYD:NM_000110:exon2:c.C85T:p.R29C,DPYD:NM_001160301:exon2:c.C85T:p.R29C	CLINSIG=pathogenic;CLNDBN=Dihydropyrimidine_dehydrogenase_deficiency;CLNACC=RCV000000464.1	rs1801265	0/1
chr1	100672060	T	C	DBT	nonsynonymous SNV	DBT:NM_001918:exon9:c.A1150G:p.S384G	CLINSIG=pathogenic;CLNDBN=I

<br>
<b><u>Format VCF for genotype imputation using TOPMed</u></b>
<br><br>
Most of the variants in your genome aren't captured by the microarray used by 23andMe/Ancestry etc.  However, you can use imputation to accurately predict the genotypes of most variants in your genome
<br><br>
In order to do this, we need to take several steps to format the VCF so that it can be uploaded to an imputation server 

First we need to strip out the 'chr' from the chromosome column in the VCF

In [ ]:
awk '{gsub(/\chr/, "")}1' my_vars.vcf > my_vars.no_chr.vcf

Next we need to compress and index the resulting VCF

In [ ]:
/opt/conda/envs/variant_calling/bin/bgzip my_vars.no_chr.vcf
/opt/conda/envs/variant_calling/bin/tabix my_vars.no_chr.vcf.gz

Finally, we need to split the VCF by chromosome

In [ ]:
/opt/conda/envs/variant_calling/bin/bcftools index -s my_vars.no_chr.vcf.gz | cut -f 1 | while read C; do /opt/conda/envs/variant_calling/bin/bcftools view -O z -o split.${C}.vcf.gz my_vars.no_chr.vcf.gz "${C}" ; done

These per-chromosome VCFs can then be uploaded to TOPMed or another imputation server (will show you how this works now)

<br>
<br>
<b>Exercise</b>
<br><br>
What disease does the individual in '23andme_indiv2.txt' have?  Look for a stopgain variant.  What does the genotype tell you about whether it is likely causal for that disease?

In [21]:
cp ~/public/variantcalling/resources/23andme_indiv2.txt .

In [ ]:
perl ~/public/variantcalling/resources/23andme2vcf.pl 23andme_indiv2.txt my_vars_2.vcf 4


In [ ]:
perl annovar/table_annovar.pl my_vars_2.vcf annovar/humandb/ -buildver hg19 -out annotated_2 -remove -protocol refGene,clinvar_20131105 -operation g,f -nastring . -vcfinput


In [ ]:
cut -f1,2,4,5,7,9,10,11,17,24 annotated_2.hg19_multianno.txt > annotated_2.hg19_multianno.trim.txt


In [ ]:
grep -v '0/0' annotated_2.hg19_multianno.trim.txt > annotated_2.hg19_multianno.trim.nohomref.txt


In [ ]:
grep '[=|]pathogenic' annotated_2.hg19_multianno.trim.nohomref.txt | grep stopgain
